# Energy Consumption Analytics Project
This notebook contains a complete data science pipeline for analyzing and predicting energy consumption based on historical usage and weather patterns.

## Project Steps
1. **Data Loading and Integration**: Merging energy consumption data with weather data.
2. **Exploratory Data Analysis (EDA)**: Descriptive statistics and visualizations.
3. **Feature Engineering**: Creating time-based features (hour, day of week) and lag features.
4. **Machine Learning Model**: Training a Random Forest Regressor to forecast future usage.
5. **Evaluation**: Assessing model performance using MAE and RMSE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set plot styles
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

## 1. Data Loading and Integration

In [ ]:
# Load the datasets generated by data_generator.py
energy_df = pd.read_csv('data/energy_data.csv', parse_dates=['timestamp'])
weather_df = pd.read_csv('data/weather_data.csv', parse_dates=['timestamp'])

# Merge datasets on timestamp
df = pd.merge(energy_df, weather_df, on='timestamp', how='inner')
df.set_index('timestamp', inplace=True)

# Handle missing values (forward fill)
df.ffill(inplace=True)

df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive Statistics
df.describe()

In [ ]:
# Plot overall energy consumption trend (weekly rolling average for smoothing)
plt.figure(figsize=(15, 6))
df['energy_consumption_kwh'].rolling(window=24*7).mean().plot()
plt.title('Energy Consumption Trend (Weekly Rolling Average)')
plt.ylabel('Energy (kWh)')
plt.xlabel('Date')
plt.show()

In [ ]:
# Correlation heat map
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 3. Feature Engineering
We extract time-based features from the index and create categorical encodings.

In [ ]:
# Time-based features
df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

# One-hot encode the weather condition
df = pd.get_dummies(df, columns=['condition'], drop_first=True)

df.head()

## 4. Machine Learning Model
We train a Random Forest model to predict `energy_consumption_kwh`.

In [ ]:
# Define Features (X) and Target (y)
target = 'energy_consumption_kwh'
features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

# Train-Test Split (Chronological to avoid data leakage)
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

In [ ]:
# Train the Random Forest Model
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained successfully!")

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error (MAE): {mae:.2f} kWh")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} kWh")

In [ ]:
# Visualize Predictions vs Actuals (First week of test data)
plt.figure(figsize=(15, 6))
plt.plot(y_test.index[:168], y_test.values[:168], label='Actual', marker='.')
plt.plot(y_test.index[:168], y_pred[:168], label='Predicted', marker='.', alpha=0.7)
plt.title('Actual vs Predicted Energy Consumption (First Week of Test Data)')
plt.ylabel('Energy (kWh)')
plt.legend()
plt.show()

In [ ]:
# Feature Importance Plot
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()